In [ ]:
from serial.tools import list_ports
import sys
import os
from src.lib.new_drivers import Bilancia, Camera
from matplotlib import pyplot as plt
import matplotlib.patches as pa
import ipywidgets as widgets
from collections import deque
import ipywidgets as widgets
from matplotlib import pyplot as plt
import os
import time
import numpy as np

In [ ]:
%matplotlib widget
plt.close('all')

In [ ]:

def make_save_dir():
    global save_dir_name
    save_dir_name = "data/raw/" + f"{time.strftime('%Y-%m-%d_%H-%M-%S')}"
    os.makedirs(save_dir_name, exist_ok=True)
    os.makedirs(save_dir_name + "/images", exist_ok=True)
    os.makedirs(save_dir_name + "/data", exist_ok=True)
    return save_dir_name

save_dir_name = None

Le camere mostrate non sono necesariamente funzionanti. Provare!
Nota: se la camera corretta è 0, anche se è gia selezionata va riselezionata dopo averne selezionata un altra.

In [ ]:
# Elenco delle porte seriali disponibili e delle camere + widget per la selezione e output dei messaggi

ports = list_ports.comports()
cameras = range(10)  # Prova a connettersi alle prime 10 camere

output_area = widgets.Output()
display(output_area)

camera = None
bilancia = None

choose_port = widgets.Dropdown(
    options=[(port.device, port.device) for port in ports],
    description='Porta:',
)

def on_port_change(change):
    global bilancia
    if change['type'] == 'change' and change['name'] == 'value':
        selected_port = change['new']
        with output_area:
            print(f"Porta selezionata: {selected_port}")
        try:
            bilancia = Bilancia(selected_port)
            bilancia.close()
            with output_area:
                print("Bilancia connessa con successo!")
        except Exception as e:
            with output_area:
                print(f"Errore durante la connessione alla bilancia: {e}")

choose_port.observe(on_port_change, names='value')

choose_camera = widgets.Dropdown(
    options=[(cam, cam) for cam in cameras],
    description='Camera:',
)

def on_camera_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_camera = change['new']
        with output_area:
            print(f"Camera selezionata: {selected_camera}")
        try:
            cam = Camera(selected_camera)
            if cam.cap.isOpened():
                with output_area:
                    print("Camera connessa con successo!")
                cam.release()
            else:
                with output_area:
                    print("Impossibile aprire la camera.")
        except Exception as e:
            with output_area:
                print(f"Errore durante la connessione alla camera: {e}")

choose_camera.observe(on_camera_change, names='value')

save_data = False

save_data_checkbox = widgets.Checkbox(
    value=False,
    description='Salva dati su file',
    disabled=False,
    indent=False
)

def on_save_data_change(change):
    global save_data
    if change['type'] == 'change' and change['name'] == 'value':
        save_data = change['new']
        with output_area:
            print(f"Salvataggio dati su file: {'Abilitato' if save_data else 'Disabilitato'}")

display(save_data_checkbox)
save_data_checkbox.observe(on_save_data_change, names='value')

display(choose_port)
display(choose_camera)

Avviare la camera con luce già accesa!!!!

In [ ]:
# Setup per visualizzare il feed della camera e la differenza tra frame e reference frame

with plt.ioff():
    camera_fig, camera_axs = plt.subplots(2, 2, figsize=(14, 8))
    display(camera_fig.canvas)
    
# inizializzo cerchio
roi_circle = pa.Circle((320, 240), radius=50, color='red', fill=False, linewidth=2)


click_state = 0  # 0 = in attesa del centro, 1 = in attesa del raggio
temp_center = (320, 240)  # Valore di default per il centro   
def on_click(event):
    global click_state, temp_center, cam, differences
    
    # Operiamo solo sul terzo subplot (la copia live)
    if event.inaxes == camera_axs[1, 0]:
        if event.xdata is None or event.ydata is None:
            return
        
        if click_state == 0:
            # STEP 1: Imposta il centro
            temp_center = (event.xdata, event.ydata)
            roi_circle.center = temp_center
            
            # Feedback visivo: cambiamo colore e stile mentre scegliamo il raggio
            roi_circle.set_edgecolor('yellow')
            roi_circle.set_linestyle('--')
            click_state = 1
            print(f"Centro fissato. Ora clicca per definire il raggio.")
            
        else:
            # STEP 2: Calcola il raggio come distanza tra centro e nuovo clic
            x2, y2 = event.xdata, event.ydata
            new_radius = np.sqrt((x2 - temp_center[0])**2 + (y2 - temp_center[1])**2)
            
            # Evitiamo raggi troppo piccoli
            if new_radius < 5: new_radius = 5 
            
            roi_circle.set_radius(new_radius)
            
            # Ripristiniamo l'aspetto normale
            roi_circle.set_edgecolor('red')
            roi_circle.set_linestyle('-')
            
            # Aggiorniamo i driver
            if cam is not None and cam.capturing:
                cam.update_roi(temp_center[0], temp_center[1], new_radius)
                print(f"ROI Aggiornata: Centro {np.round(temp_center,1)}, Raggio {new_radius:.1f}")
            
            click_state = 0 # Torna allo stato iniziale per un'eventuale nuova modifica
        
        differences = []
            
        camera_fig.canvas.draw_idle()
            

camera_fig.canvas.mpl_connect('button_press_event', on_click)

camera_running = False

start_camera_button = widgets.Button(description="Start Camera")
stop_camera_button = widgets.Button(description="Stop Camera")

cam = None

camera_img_artist = None
camera_img_artist_diff = None
camera_img_artist_copy = None 
camera_img_timer = None
roi_artist = None

int_timer = None

with plt.ioff():
    fig_int, axs_int = plt.subplots(1, 1, figsize=(14, 6))

    axs_int = [axs_int]

    int_line, = axs_int[0].plot([], [], label="Spessori", marker="o")

    axs_int[0].set_title("Integrale differenze")
    axs_int[0].set_xlabel("Time")
    axs_int[0].set_ylabel("Integrale")
    axs_int[0].grid()

def update_plot_int():
    global differences, int_line, fig_int, axs_int

    if not camera_running:
        return

    if len(differences) == 0:
        return

    y = np.asarray(differences, dtype=float)
    y = y[np.isfinite(y)]
    if y.size == 0:
        return

    x = np.arange(y.size)
    int_line.set_data(x, y)

    if y.size == 1:
        axs_int[0].set_xlim(x[0] - 1, x[0] + 1)
        pad = max(abs(y[0]) * 0.1, 1e-6)
        axs_int[0].set_ylim(y[0] - pad, y[0] + pad)
    else:
        axs_int[0].set_xlim(x.min(), x.max())
        ymin, ymax = y.min(), y.max()
        pad = max((ymax - ymin) * 0.1, 1e-6)
        axs_int[0].set_ylim(ymin - pad, ymax + pad)

    fig_int.canvas.draw_idle()

def start_camera(b):
    global camera_running, cam, camera_img_timer, camera_img_artist, camera_img_artist_diff, camera_img_artist_copy, save_dir_name, int_timer, last_frames, differences, roi_artist

    if save_data:
        make_save_dir()  # Crea una nuova cartella per ogni sessione di acquisizione

    try:
        cam = Camera(choose_camera.value, keep_frames=1)
        if cam.cap.isOpened():
            with output_area:
                print("Camera connessa con successo!")
        else:
            with output_area:
                print("Impossibile aprire la camera.")
    except Exception as e:
        with output_area:
            print(f"Errore durante la connessione alla camera: {e}")
        cam = None

    if cam is not None:
        cam._acquire_reference_image()
        cam.start_acquisition(center_x=320, center_y=240, radius=100, interval=0.1)

        # Crea artist una sola volta
        if camera_img_artist is None:
            camera_img_artist = camera_axs[0, 0].imshow(np.zeros((480, 640)), cmap="gray")
            camera_axs[0, 0].set_title("Live camera")
            camera_axs[0, 0].axis("off")

        if camera_img_artist_diff is None:
            camera_img_artist_diff = camera_axs[0, 1].imshow(np.zeros((480, 640)), cmap="gray")
            camera_axs[0, 1].set_title("Difference")
            camera_axs[0, 1].axis("off")
            
        if camera_img_artist_copy is None:
            camera_img_artist_copy = camera_axs[1, 0].imshow(np.zeros((480, 640)), cmap="gray")
            camera_axs[1, 0].set_title("ROI Dinamica (Clicca per muovere)")
            camera_axs[1, 0].axis("off")
            camera_axs[1, 0].add_patch(roi_circle)
        
        if roi_artist is None:
            roi_artist = camera_axs[1, 1].imshow(np.zeros((480, 640)), cmap="gray")
            camera_axs[1, 1].set_title("ROI")
            camera_axs[1, 1].axis("off")

        camera_img_timer = camera_fig.canvas.new_timer(interval=100)  # Aggiorna ogni 100 ms
        camera_img_timer.add_callback(update_frame_camera)
        camera_img_timer.start()

        int_timer = fig_int.canvas.new_timer(interval=500)  # Aggiorna ogni 500 ms
        int_timer.add_callback(update_plot_int)
        int_timer.start() 

        camera_running = True
        start_camera_button.disabled = True
        stop_camera_button.disabled = False

        if save_data:
            # save the reference frame (first frame) to the save_dir_name/data folder with timestamp
            timestamp_str = time.strftime("%Y%m%d_%H%M%S")
            reference_frame_filename = f"{save_dir_name}/reference_frame_{timestamp_str}.png"
            plt.imsave(reference_frame_filename, cam.im0, cmap="gray")

    else:
        with output_area:
            print("Nessuna camera selezionata.")

def stop_camera(b):
    global camera_running, cam, camera_img_timer, camera_img_artist, camera_img_artist_diff, camera_img_artist_copy, int_timer, roi_artist
    camera_running = False

    if camera_img_timer is not None:
        camera_img_timer.stop()
        camera_img_timer = None
        int_timer.stop()
        int_timer = None

    if cam is not None:
        cam.stop_acquisition()
        cam.release()
        cam = None
        start_camera_button.disabled = False
        stop_camera_button.disabled = True
    else:
        with output_area:
            print("Nessuna camera selezionata.")

last_frames = []
differences = []
save_img_counter = 0

def update_frame_camera():
    global cam, camera_img_artist, camera_img_artist_diff, camera_img_artist_copy, save_dir_name, save_img_counter, differences, last_frames, roi_artist

    if cam is None or not cam.cap.isOpened():
        return

    sample = cam.get_latest_image()
    if sample is None:
        return

    data, ts = sample
    frame = data[0] 
    roi = data[1]

    cam.images.clear()  # Pulisce il buffer delle immagini dopo aver preso l'ultima
    cam.timestamps.clear()  # Pulisce il buffer dei timestamp

    if frame is not None:
        frames = np.array(last_frames)
        last_frames.append(frame)
        avg_frame = np.mean(frames, axis=0)

        #avg_frame = np.array(avg_frame, dtype=np.uint32)

        if len(last_frames) > 10:
            last_frames.pop(0)

    if frame is not None and camera_img_artist is not None:
        camera_img_artist.set_data(frame)
        camera_img_artist.set_clim(vmin=float(frame.min()), vmax=float(frame.max()) + 1e-6)

        if camera_img_artist_copy is not None:
            camera_img_artist_copy.set_data(frame)
            camera_img_artist_copy.set_clim(vmin=float(frame.min()), vmax=float(frame.max()) + 1e-6)
    
        save_img_counter += 1
        if save_img_counter % 10 == 0 and save_data:
            # save image to the save_dir_name/images folder with timestamp
            timestamp_str = time.strftime("%Y%m%d_%H%M%S")
            image_filename = f"{save_dir_name}/images/frame_{timestamp_str}.png"
            plt.imsave(image_filename, frame, cmap="gray")
        
        if save_img_counter > 1000:
            save_img_counter = 0

    if frame is not None and camera_img_artist_diff is not None:
        im0 = np.array(cam.im0)
        diff = abs(avg_frame - im0)
        camera_img_artist_diff.set_data(diff)
        camera_img_artist_diff.set_clim(vmin=float(diff.min()), vmax=float(diff.max()) + 1e-6)
    
    
    if roi_artist is not None and roi is not None:
        cut_diff = diff * cam.masks['total']
        differences.append(np.mean(cut_diff))
        roi_artist.set_data(cut_diff)
        min_nonzero = cut_diff[cut_diff > 0].min()
        max_nonzero = cut_diff[cut_diff > 0].max()
        roi_artist.set_clim(vmin=float(min_nonzero), vmax=float(max_nonzero) + 1e-6)
    
    if frame is not None:
        camera_fig.canvas.draw_idle()

start_camera_button.on_click(start_camera)
stop_camera_button.on_click(stop_camera)
display(start_camera_button)
display(stop_camera_button)


In [ ]:
display(fig_int.canvas)

In [ ]:
# Grafico per spessori e rate in tempo reale

spessori = deque(maxlen=100)
rate = deque(maxlen=100)
timestamps = deque(maxlen=100)

Running = False
plot_timer = None

t_output = widgets.Output()
display(t_output)

with plt.ioff():
    fig, axs = plt.subplots(1, 2, figsize=(14, 6))
    display(fig.canvas)

    spessore_line, = axs[0].plot([], [], label="Spessori", marker="o")
    rate_line, = axs[1].plot([], [], label="Rate", marker="o")

    axs[0].set_title("Rate")
    axs[0].set_xlabel("Time")
    axs[0].set_ylabel("Spessore")
    axs[0].grid()

    axs[1].set_title("Spessore")
    axs[1].set_xlabel("Time")
    axs[1].set_ylabel("Rate")
    axs[1].grid()

stop_btn = widgets.Button(description="Stop", disabled=True)
start_btn = widgets.Button(description="Start", disabled=False)
display(start_btn, stop_btn)

def reset_plot():
    spessori.clear()
    rate.clear()
    timestamps.clear()
    spessore_line.set_data([], [])
    rate_line.set_data([], [])
    fig.canvas.draw_idle()

def _safe_ylim(values):
    if not values:
        return (-1, 1)

    vmin = min(values)
    vmax = max(values)

    # Caso serie costante: crea comunque un intervallo visibile
    if vmin == vmax:
        pad = max(abs(vmin) * 0.2, 1e-6)
        return (vmin - pad, vmax + pad)

    # Padding del 10% rispetto allo span reale
    span = vmax - vmin
    pad = max(span * 0.1, 1e-6)
    return (vmin - pad, vmax + pad)

save_counter = 0

def update_plot_once():
    global save_counter, save_data, save_dir_name, spessore_line, rate_line, fig, axs, bilancia
    t = bilancia.get_latest_data()
    if t is None:
        return

    try:
        (spessore, r), timestamp = t
        spessore = float(spessore)
        r = float(r)
        timestamp = float(timestamp)

    except (TypeError, ValueError):
        # Scarta campione corrotto, non uccidere l'update loop
        with output_area:
            print("Dati corrotti ricevuti, scartando campione")
        return
    
    save_counter += 1

    if save_counter % 10 == 0 and save_data:
        # append the data to the end of a file in save_dir_name/data/points.txt with timestamp

        data_filename = f"{save_dir_name}/data/points.txt"
        with open(data_filename, "a") as f:
            f.write(f"{timestamp}\t{spessore}\t{r}\n")
    
    if save_counter > 1000:
        save_counter = 0


    spessori.append(spessore)
    rate.append(r)
    timestamps.append(timestamp)

    if len(timestamps) > 10000:
        timestamps.popleft()
        spessori.popleft()
        rate.popleft()

    x = list(timestamps)
    y1 = list(spessori)
    y2 = list(rate)

    spessore_line.set_data(x, y1)
    rate_line.set_data(x, y2)

    if len(x) == 0:
        axs[0].set_xlim(0, 1)
        axs[1].set_xlim(0, 1)

    if len(x) == 1:
        axs[0].set_xlim(x[0] - 1, x[0] + 1)
        axs[1].set_xlim(x[0] - 1, x[0] + 1)
    elif len(x) > 1:
        axs[0].set_xlim(min(x), max(x))
        axs[1].set_xlim(min(x), max(x))

    axs[0].set_ylim(*_safe_ylim(y1))
    axs[1].set_ylim(*_safe_ylim(y2))

    fig.canvas.draw_idle()

def on_start_btn_click(b):
    global Running, plot_timer, bilancia, save_dir_name
    if Running:
        return

    bilancia = Bilancia(choose_port.value)
    save_dir_name = make_save_dir()

    reset_plot()
    bilancia.start_continuous_read(data=["Sensor 1 rate", "Sensor 1 thickness"])

    plot_timer = fig.canvas.new_timer(interval=100)
    plot_timer.add_callback(update_plot_once)
    plot_timer.start()

    Running = True
    start_btn.disabled = True
    stop_btn.disabled = False

def on_stop_btn_click(b):
    global Running, plot_timer
    if not Running:
        return

    Running = False

    if plot_timer is not None:
        plot_timer.stop()
        plot_timer = None

    bilancia.stop_continuous_read()
    start_btn.disabled = False
    stop_btn.disabled = True

    bilancia.close()

def on_close(evt):
    # Chiusura pulita se chiudi la figura
    if Running:
        on_stop_btn_click(None)

fig.canvas.mpl_connect("close_event", on_close)

start_btn.on_click(on_start_btn_click)
stop_btn.on_click(on_stop_btn_click)